<a href="https://colab.research.google.com/github/akira-kun/GY-MAX4466-ESP32/blob/master/SoulX_FlashHead_Lite_Colab_touch2_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎭 SoulX FlashHead Lite — Google Colab

**Geração de talking head com lip sync, regenerando pixels reais (sem warping 2D)**

**Requisitos oficiais do projeto:**
- Python 3.10
- PyTorch 2.7.1 + CUDA 12.8
- flash_attn 2.8.0.post2 (requer Ampere+)
- sageattention 2.2.0

**O notebook detecta automaticamente a GPU e adapta a instalação:**
- 🚀 **T4 / A100 / RTX 30XX+** → instalação completa com flash_attn
- ⚠️ **RTX 2060 / Turing consumer** → flash_attn substituído por SDPA (torch nativo)

**Nada é salvo no Google Drive.** O vídeo é baixado direto para o seu PC.

---
⚠️ **Antes de começar:** `Ambiente de execução → Alterar tipo → GPU (T4)`

▶️ **Execute as células em ordem.**

In [1]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 1 — Detectar GPU e definir estratégia de instalação
# ═══════════════════════════════════════════════════════════════
import subprocess, sys

def detectar_gpu():
    result = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,compute_cap,memory.total', '--format=csv,noheader'],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError('Nenhuma GPU detectada! Ative a GPU nas configuracoes do Colab.')

    linha = result.stdout.strip().split('\n')[0].split(',')
    nome   = linha[0].strip()
    cap    = linha[1].strip()         # ex: '7.5' = Turing, '8.0' = Ampere
    vram   = linha[2].strip()         # ex: '15360 MiB'
    major  = int(cap.split('.')[0])

    # Turing consumer = compute 7.x + nome de GPU gaming
    TURING_IDS = ['2060','2070','2080','1660','1650']
    is_turing_consumer = major == 7 and any(x in nome for x in TURING_IDS)

    # flash_attn 2.x requer Ampere (8.x) ou superior
    # T4 = compute 7.5 mas e datacenter, tem suporte parcial
    suporta_flash_attn2 = (major >= 8) or (major == 7 and 'T4' in nome)

    return nome, cap, vram, is_turing_consumer, suporta_flash_attn2

GPU_NOME, GPU_COMPUTE, GPU_VRAM, IS_TURING_CONSUMER, USA_FLASH_ATTN = detectar_gpu()

print('=' * 60)
print(f'  GPU          : {GPU_NOME}')
print(f'  Compute Cap. : {GPU_COMPUTE}')
print(f'  VRAM         : {GPU_VRAM}')
print('=' * 60)

if IS_TURING_CONSUMER:
    print('  Modo         : ⚠️  TURING CONSUMER')
    print('  flash_attn   : DESATIVADO (nao suportado na RTX 20XX)')
    print('  Atencao      : SDPA (PyTorch nativo) — mais lento, mas funciona')
    print('  sageattention: v1.0.6 (versao Turing)')
elif USA_FLASH_ATTN:
    print('  Modo         : 🚀 AMPERE / ADA / T4')
    print('  flash_attn   : ATIVADO (2.8.0.post2)')
    print('  sageattention: v2.2.0 (versao completa)')
else:
    print('  Modo         : GPU nao reconhecida — tentando instalacao padrao')

print('=' * 60)
print('\n✅ Deteccao concluida! Execute a proxima celula.')

  GPU          : Tesla T4
  Compute Cap. : 7.5
  VRAM         : 15360 MiB
  Modo         : 🚀 AMPERE / ADA / T4
  flash_attn   : ATIVADO (2.8.0.post2)
  sageattention: v2.2.0 (versao completa)

✅ Deteccao concluida! Execute a proxima celula.


In [2]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 2 — Verificar/ajustar versão do Python
# O SoulX FlashHead requer Python 3.10 especificamente
# ═══════════════════════════════════════════════════════════════
import sys

versao = sys.version_info
print(f'Python detectado: {versao.major}.{versao.minor}.{versao.micro}')

if versao.major == 3 and versao.minor == 10:
    print('✅ Python 3.10 confirmado — compativel com SoulX FlashHead!')
elif versao.major == 3 and versao.minor == 11:
    print('⚠️  Python 3.11 detectado.')
    print('   O SoulX FlashHead foi desenvolvido para 3.10, mas 3.11 geralmente funciona.')
    print('   Continuando... Se houver erros de compatibilidade, troque o runtime para Python 3.10.')
else:
    print(f'⚠️  Python {versao.major}.{versao.minor} detectado.')
    print('   Recomendado: Python 3.10. Pode haver incompatibilidades.')

# Verificar CUDA disponivel
import subprocess
result = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
if result.returncode == 0:
    linha_versao = [l for l in result.stdout.split('\n') if 'release' in l.lower()]
    print(f'\nCUDA toolkit: {linha_versao[0].strip() if linha_versao else "detectado"}')
else:
    print('\nCUDA toolkit: nao encontrado no PATH (normal no Colab — PyTorch tem CUDA embutido)')

Python detectado: 3.12.13
⚠️  Python 3.12 detectado.
   Recomendado: Python 3.10. Pode haver incompatibilidades.

CUDA toolkit: Cuda compilation tools, release 12.8, V12.8.93


In [2]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 3 — Clonar repositório
# ═══════════════════════════════════════════════════════════════
import os

%cd /content

if not os.path.exists('/content/SoulX-FlashHead'):
    print('Clonando SoulX-FlashHead...')
    !git clone -q https://github.com/Soul-AILab/SoulX-FlashHead.git
    print('✅ Repositorio clonado!')
else:
    print('✅ Repositorio ja existe nesta sessao.')
    print('   Atualizando...')
    !cd /content/SoulX-FlashHead && git pull -q

%cd /content/SoulX-FlashHead
print(f'\nDiretorio: {os.getcwd()}')
!ls -la

/content
✅ Repositorio ja existe nesta sessao.
   Atualizando...
/content/SoulX-FlashHead

Diretorio: /content/SoulX-FlashHead
total 116
drwxr-xr-x 6 root root  4096 Apr 23 18:36 .
drwxr-xr-x 1 root root  4096 Apr 23 18:36 ..
drwxr-xr-x 2 root root  4096 Apr 23 18:36 assets
drwxr-xr-x 2 root root  4096 Apr 23 18:36 examples
drwxr-xr-x 8 root root  4096 Apr 23 18:36 flash_head
-rw-r--r-- 1 root root  8666 Apr 23 18:36 generate_video.py
drwxr-xr-x 8 root root  4096 Apr 23 18:37 .git
-rw-r--r-- 1 root root   126 Apr 23 18:36 .gitignore
-rw-r--r-- 1 root root 17028 Apr 23 18:36 gradio_app.py
-rw-r--r-- 1 root root 12761 Apr 23 18:36 gradio_app_streaming.py
-rw-r--r-- 1 root root   395 Apr 23 18:36 inference_script_multi_gpu_pro.sh
-rw-r--r-- 1 root root   328 Apr 23 18:36 inference_script_single_gpu_lite.sh
-rw-r--r-- 1 root root   327 Apr 23 18:36 inference_script_single_gpu_pro.sh
-rw-r--r-- 1 root root 11357 Apr 23 18:36 LICENSE
-rw-r--r-- 1 root root 11431 Apr 23 18:36 README.md
-rw-r-

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 4 — Instalar PyTorch + CUDA 12.8
# Versao exigida pelo SoulX FlashHead: torch 2.7.1 + cu128
# ═══════════════════════════════════════════════════════════════
import torch, importlib

torch_atual = torch.__version__
print(f'PyTorch atual: {torch_atual}')

# Verificar se ja e a versao correta
if '2.7' in torch_atual and 'cu12' in torch_atual:
    print('✅ PyTorch 2.7.x com CUDA 12 ja instalado!')
else:
    print('Instalando PyTorch 2.7.1 + CUDA 12.8...')
    print('Isso pode demorar 2-3 minutos...\n')
    !pip install -q torch==2.7.1 torchvision==0.22.1 \
        --index-url https://download.pytorch.org/whl/cu128
    print('\n✅ PyTorch 2.7.1 + cu128 instalado!')
    print('   Reiniciando kernel para aplicar...')
    import os
    os.kill(os.getpid(), 9)  # Reinicia o kernel automaticamente

PyTorch atual: 2.10.0+cu128
Instalando PyTorch 2.7.1 + CUDA 12.8...
Isso pode demorar 2-3 minutos...

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 120.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 254.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 128.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 726.9/726.9 MB 58.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 609.6/609.6 MB 45.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 101.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 113.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.4/260.4 MB 83.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 292.1/292.1 MB 87.9 MB/s eta 0:00:00
     ━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/156.8 MB 163.5 MB/s eta 0:00:01
ERROR: Operation cancelled by user
^C


In [23]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 5 — Instalar dependências base (requirements.txt)
# ═══════════════════════════════════════════════════════════════
import os
%cd /content/SoulX-FlashHead

print('Instalando dependencias base (requirements.txt)...')
print('Pode demorar 3-5 minutos...\n')

!apt-get install -q ffmpeg
!pip install -q ninja

with open("requirements.txt", "r") as f:
    content = f.read()

content = content.replace("mediapipe==0.10.9", "mediapipe==0.10.20")
content = content.replace("nvidia-nccl-cu12==2.27.3", "")
content = content.replace("xformers==0.0.31", "")
content = content.replace("opencv-python>=4.12.0.88", "opencv-python==4.8.1.78")
content = content.replace("opencv-python-headless>=4.12.0.88", "")

with open("requirements.txt", "w") as f:
    f.write(content)

!pip install -q -r requirements.txt

print('\n✅ Dependencias base instaladas!')

/content/SoulX-FlashHead
Instalando dependencias base (requirements.txt)...
Pode demorar 3-5 minutos...

Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 126.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 241.4/241.4 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 110.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0

In [4]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 6 — Instalar flash_attn e sageattention
# Adapta automaticamente para cada GPU
# ═══════════════════════════════════════════════════════════════
import subprocess, sys, os

# Re-detectar GPU (necessario apos reinicio de kernel)
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,compute_cap', '--format=csv,noheader'],
    capture_output=True, text=True
)
linha = result.stdout.strip().split('\n')[0].split(',')
GPU_NOME = linha[0].strip()
GPU_COMPUTE = linha[1].strip()
major = int(GPU_COMPUTE.split('.')[0])
TURING_IDS = ['2060','2070','2080','1660','1650']
IS_TURING_CONSUMER = major == 7 and any(x in GPU_NOME for x in TURING_IDS)
USA_FLASH_ATTN = (major >= 8) or (major == 7 and 'T4' in GPU_NOME)

print(f'GPU: {GPU_NOME} (compute {GPU_COMPUTE})')
print('=' * 50)

if IS_TURING_CONSUMER:
    # RTX 2060 etc: flash_attn 2.x nao suporta Turing consumer
    # Usar sageattention v1 + SDPA como fallback
    print('⚠️  RTX 2060/Turing consumer detectado.')
    print('   flash_attn 2.x requer Ampere+ — PULANDO instalacao.')
    print('   Sera usado SDPA (PyTorch Scaled Dot Product Attention) como alternativa.')
    print('\nInstalando sageattention v1.0.6 (compativel com Turing)...')
    ret = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', 'sageattention==1.0.6'],
        capture_output=True, text=True
    )
    if ret.returncode == 0:
        print('✅ sageattention 1.0.6 instalado!')
    else:
        print('⚠️  sageattention nao instalou — continuando sem ele (mais lento).')
    ATTN_MODE = 'sdpa'

elif USA_FLASH_ATTN:
    print('🚀 GPU compativel com flash_attn 2.x detectada!')
    print('   Instalando flash_attn 2.8.0.post2...')
    print('   AVISO: pode demorar 5-15 minutos (compila do zero).')
    print('   Alternativa rapida: baixar wheel pre-compilado.\n')

    # Tentar instalar wheel pre-compilado primeiro (muito mais rapido)
    # Wheel oficial para Python 3.10 + CUDA 12 + torch 2.7
    WHEEL_URL = (
        'https://github.com/Dao-AILab/flash-attention/releases/download/'
        'v2.8.0.post2/flash_attn-2.8.0.post2+cu12torch2.7cxx11abiFALSE'
        '-cp310-cp310-linux_x86_64.whl'
    )
    print(f'Tentando wheel pre-compilado...')
    ret = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', WHEEL_URL],
        capture_output=True, text=True, timeout=300
    )
    if ret.returncode == 0:
        print('✅ flash_attn instalado via wheel pre-compilado!')
    else:
        print('   Wheel pre-compilado falhou. Compilando do zero (~10 min)...')
        !pip install flash_attn==2.8.0.post2 --no-build-isolation

    print('\nInstalando sageattention 2.2.0...')
    !pip install sageattention==2.2.0 --no-build-isolation
    print('✅ sageattention 2.2.0 instalado!')
    ATTN_MODE = 'flash_attn'

else:
    print('GPU nao reconhecida — tentando instalacao padrao...')
    !pip install flash_attn==2.8.0.post2 --no-build-isolation
    !pip install sageattention==2.2.0 --no-build-isolation
    ATTN_MODE = 'flash_attn'

print(f'\n✅ Modo de atencao definido: {ATTN_MODE.upper()}')

GPU: Tesla T4 (compute 7.5)
🚀 GPU compativel com flash_attn 2.x detectada!
   Instalando flash_attn 2.8.0.post2...
   AVISO: pode demorar 5-15 minutos (compila do zero).
   Alternativa rapida: baixar wheel pre-compilado.

Tentando wheel pre-compilado...
   Wheel pre-compilado falhou. Compilando do zero (~10 min)...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 89.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for flash_attn
  Running setup.py clean for flash_attn
Failed to build flash_attn
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (flash_attn)

Instalando sageattention 2.2.0...
ERROR: Could not find a version that satisfies the requirement sageattention==2.

In [14]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 7 — Baixar pesos do modelo Lite
# Ficam em /content/ — NAO ocupam o Google Drive
# Tamanho: ~5-8GB — pode demorar 5-10 min
# ═══════════════════════════════════════════════════════════════
import os

MODELS_DIR = '/content/SoulX-FlashHead/models'
FLASHHEAD_DIR = f'{MODELS_DIR}/SoulX-FlashHead-1_3B'
WAV2VEC_DIR = f'{MODELS_DIR}/wav2vec2-base-960h'

!pip install -q huggingface_hub

os.makedirs(MODELS_DIR, exist_ok=True)

# Baixar modelo FlashHead
if not os.path.exists(FLASHHEAD_DIR) or not os.listdir(FLASHHEAD_DIR):
    print('Baixando SoulX-FlashHead-1_3B (~5-8GB)...')
    print('Os pesos ficam em /content/ — nao ocupam seu Google Drive.')
    print('Pode demorar 5-10 minutos...\n')
    !hf download Soul-AILab/SoulX-FlashHead-1_3B \
        --local-dir {FLASHHEAD_DIR}
    print('\n✅ Modelo FlashHead baixado!')
else:
    print('✅ Modelo FlashHead ja existe nesta sessao.')

# Baixar wav2vec2 (encoder de audio)
if not os.path.exists(WAV2VEC_DIR) or not os.listdir(WAV2VEC_DIR):
    print('\nBaixando wav2vec2-base-960h (~400MB)...')
    !hf download facebook/wav2vec2-base-960h \
        --local-dir {WAV2VEC_DIR}
    print('✅ wav2vec2 baixado!')
else:
    print('✅ wav2vec2 ja existe nesta sessao.')

# Listar arquivos do modelo Lite especificamente
print('\nArquivos do modelo disponíveis:')
!ls -lh {FLASHHEAD_DIR}/ 2>/dev/null | head -20

Baixando SoulX-FlashHead-1_3B (~5-8GB)...
Os pesos ficam em /content/ — nao ocupam seu Google Drive.
Pode demorar 5-10 minutos...

Fetching 18 files:   0% 0/18 [00:00<?, ?it/s]Still waiting to acquire lock on /content/SoulX-FlashHead/models/SoulX-FlashHead-1_3B/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Fetching 18 files: 100% 18/18 [01:51<00:00,  6.17s/it]
Download complete: : 14.3GB [01:51, 150MB/s]              ✓ Downloaded
  path: /content/SoulX-FlashHead/models/SoulX-FlashHead-1_3B
Download complete: : 14.3GB [01:51, 129MB/s]

✅ Modelo FlashHead baixado!

Baixando wav2vec2-base-960h (~400MB)...
Fetching 11 files:   0% 0/11 [00:00<?, ?it/s]Still waiting to acquire lock on /content/SoulX-FlashHead/models/wav2vec2-base-960h/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Fetching 11 files: 100% 11/11 [00:09<00:00,  1.15it/s]
Download complete: : 1.13GB [00:09, 185MB/s]              ✓ Downloaded
  path: /content/SoulX-FlashHead/models/wav2vec2-base-960h
Downlo

In [6]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 8 — Patch automático para RTX 2060 (desativa flash_attn)
# Substitui flash_attn por SDPA nos arquivos do modelo
# Esta célula é IGNORADA automaticamente em GPUs Ampere+
# ═══════════════════════════════════════════════════════════════
import subprocess, os

# Re-detectar GPU
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,compute_cap', '--format=csv,noheader'],
    capture_output=True, text=True
)
linha = result.stdout.strip().split('\n')[0].split(',')
GPU_NOME = linha[0].strip()
major = int(linha[1].strip().split('.')[0])
TURING_IDS = ['2060','2070','2080','1660','1650']
IS_TURING_CONSUMER = major == 7 and any(x in GPU_NOME for x in TURING_IDS)

if not IS_TURING_CONSUMER:
    print(f'✅ GPU {GPU_NOME} — flash_attn suportado. Nenhum patch necessario.')
else:
    print(f'⚙️  Aplicando patch de compatibilidade para {GPU_NOME}...')
    print('   Substituindo flash_attn por SDPA nos arquivos do modelo...\n')

    # Patch: substituir use_flash_attention=True por False em configs
    import glob
    arquivos_py = glob.glob('/content/SoulX-FlashHead/flash_head/**/*.py', recursive=True)
    arquivos_py += glob.glob('/content/SoulX-FlashHead/*.py')

    patches_aplicados = 0
    for arq in arquivos_py:
        try:
            with open(arq, 'r', encoding='utf-8') as f:
                conteudo = f.read()
            novo = conteudo
            # Substituicoes comuns de flash_attn por SDPA
            if 'flash_attn' in conteudo or 'use_flash_attention' in conteudo:
                novo = conteudo.replace(
                    'from flash_attn import flash_attn_func',
                    '# flash_attn desativado para Turing — usando SDPA\n'
                    'import torch\n'
                    'def flash_attn_func(q, k, v, dropout_p=0.0, softmax_scale=None, causal=False, **kwargs):\n'
                    '    scale = softmax_scale or (q.shape[-1] ** -0.5)\n'
                    '    return torch.nn.functional.scaled_dot_product_attention(q.transpose(1,2), k.transpose(1,2), v.transpose(1,2), scale=scale, is_causal=causal).transpose(1,2)'
                )
                novo = novo.replace(
                    'from flash_attn.flash_attn_interface import flash_attn_varlen_func',
                    '# flash_attn_varlen desativado para Turing'
                )
                novo = novo.replace('use_flash_attention=True', 'use_flash_attention=False')
                novo = novo.replace('use_flash_attn=True', 'use_flash_attn=False')
                if novo != conteudo:
                    with open(arq, 'w', encoding='utf-8') as f:
                        f.write(novo)
                    patches_aplicados += 1
                    print(f'   Patched: {os.path.basename(arq)}')
        except Exception as e:
            pass

    if patches_aplicados > 0:
        print(f'\n✅ Patch aplicado em {patches_aplicados} arquivo(s).')
    else:
        print('\nℹ️  Nenhum arquivo precisou de patch direto.')
        print('   O modelo pode usar flash_attn via try/except — testando na geracao.')

    print('\n⚠️  NOTA: SDPA e mais lento que flash_attn (~30-50% mais lento).')
    print('   No Colab T4, o flash_attn funciona. Na RTX 2060 local, use este patch.')

✅ GPU Tesla T4 — flash_attn suportado. Nenhum patch necessario.


In [7]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 9 — Upload da IMAGEM (foto/avatar)
# ═══════════════════════════════════════════════════════════════
from google.colab import files
import os, cv2

print('📁 Faca upload da sua IMAGEM (avatar — JPG ou PNG)')
print('   Dicas para melhor resultado:')
print('   ✔ Rosto frontal ou ate 3/4')
print('   ✔ Boa iluminacao uniforme')
print('   ✔ Rosto ocupa 50-80% da imagem')
print('   ✔ Resolucao minima: 256x256 (ideal: 512x512+)\n')
print('   Diferenca do LivePortrait: este modelo REGENERA pixels reais!')
print('   Nao ha "esticamento" de pele — novos pixels sao gerados por difusao.\n')

uploaded = files.upload()
os.makedirs('/content/inputs', exist_ok=True)

for filename, content in uploaded.items():
    ext = filename.split('.')[-1].lower()
    dest = f'/content/inputs/avatar.{ext}'
    with open(dest, 'wb') as f:
        f.write(content)
    SOURCE_IMAGE = dest
    print(f'\n✅ Imagem salva: {dest} ({len(content)/1024:.0f} KB)')
    break

try:
    img = cv2.imread(SOURCE_IMAGE)
    h, w = img.shape[:2]
    status = '✅ OK' if w >= 256 and h >= 256 else '⚠️  Muito pequena (min 256x256)'
    print(f'   Resolucao: {w}x{h} — {status}')
    if w < 256 or h < 256:
        print('   Redimensionando para 512x512...')
        img_res = cv2.resize(img, (512, 512))
        cv2.imwrite(SOURCE_IMAGE, img_res)
        print('   ✅ Imagem redimensionada!')
except Exception as e:
    print(f'   (Nao foi possivel verificar resolucao: {e})')

print(f'\n🖼️  Avatar: {SOURCE_IMAGE}')

📁 Faca upload da sua IMAGEM (avatar — JPG ou PNG)
   Dicas para melhor resultado:
   ✔ Rosto frontal ou ate 3/4
   ✔ Boa iluminacao uniforme
   ✔ Rosto ocupa 50-80% da imagem
   ✔ Resolucao minima: 256x256 (ideal: 512x512+)

   Diferenca do LivePortrait: este modelo REGENERA pixels reais!
   Nao ha "esticamento" de pele — novos pixels sao gerados por difusao.



Saving mulher-512-enhace.png to mulher-512-enhace.png

✅ Imagem salva: /content/inputs/avatar.png (311 KB)
   Resolucao: 512x512 — ✅ OK

🖼️  Avatar: /content/inputs/avatar.png


In [12]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 10 — Upload do ÁUDIO (WAV ou MP3)
# ═══════════════════════════════════════════════════════════════
from google.colab import files
import os, subprocess

print('📁 Faca upload do AUDIO (WAV ou MP3)')
print('   Dicas:')
print('   ✔ Qualquer lingua funciona — PT-BR incluido!')
print('   ✔ Voz clara, sem muito eco ou ruido de fundo')
print('   ✔ Para teste inicial, use clips de 5-15 segundos')
print('   ✔ O modelo sincroniza labios com o audio automaticamente\n')

uploaded_audio = files.upload()
os.makedirs('/content/inputs', exist_ok=True)

for filename, content in uploaded_audio.items():
    ext = filename.split('.')[-1].lower()
    dest_raw = f'/content/inputs/audio_raw.{ext}'
    with open(dest_raw, 'wb') as f:
        f.write(content)
    print(f'\n✅ Audio salvo: {dest_raw} ({len(content)/1024:.0f} KB)')
    break

# Converter para WAV 16kHz mono (formato esperado pelo wav2vec2)
AUDIO_WAV = '/content/inputs/audio.wav'
print('   Convertendo para WAV 16kHz mono (requerido pelo wav2vec2)...')
ret = subprocess.run(
    ['ffmpeg', '-y', '-i', dest_raw, '-ar', '16000', '-ac', '1', '-acodec', 'pcm_s16le', AUDIO_WAV],
    capture_output=True, text=True
)
if ret.returncode == 0:
    duracao_result = subprocess.run(
        ['ffprobe', '-v', 'quiet', '-show_entries', 'format=duration',
         '-of', 'csv=p=0', AUDIO_WAV],
        capture_output=True, text=True
    )
    try:
        dur = float(duracao_result.stdout.strip())
        print(f'   ✅ Audio convertido: {dur:.1f}s de audio a 16kHz mono')
    except:
        print('   ✅ Audio convertido!')
else:
    print(f'   ⚠️  Erro na conversao: {ret.stderr[-200:]}')
    AUDIO_WAV = dest_raw
    print(f'   Usando audio original: {AUDIO_WAV}')

print(f'\n🎵 Audio: {AUDIO_WAV}')

📁 Faca upload do AUDIO (WAV ou MP3)
   Dicas:
   ✔ Qualquer lingua funciona — PT-BR incluido!
   ✔ Voz clara, sem muito eco ou ruido de fundo
   ✔ Para teste inicial, use clips de 5-15 segundos
   ✔ O modelo sincroniza labios com o audio automaticamente



Saving suas-3b.mp3 to suas-3b (1).mp3

✅ Audio salvo: /content/inputs/audio_raw.mp3 (198 KB)
   Convertendo para WAV 16kHz mono (requerido pelo wav2vec2)...
   ✅ Audio convertido: 16.8s de audio a 16kHz mono

🎵 Audio: /content/inputs/audio.wav


In [15]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 11 — Configurações de geração
# ═══════════════════════════════════════════════════════════════
import os, subprocess

# Re-detectar GPU para configuracoes
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,compute_cap', '--format=csv,noheader'],
    capture_output=True, text=True
)
linha = result.stdout.strip().split('\n')[0].split(',')
GPU_NOME = linha[0].strip()
major = int(linha[1].strip().split('.')[0])
TURING_IDS = ['2060','2070','2080','1660','1650']
IS_TURING_CONSUMER = major == 7 and any(x in GPU_NOME for x in TURING_IDS)

# ─────────────────────────────────────────────────────────────
# Pasta de saida na memoria temporaria do Colab — nao vai para o Drive
OUTPUT_DIR = '/content/outputs'

# Caminho dos modelos
MODEL_PATH = '/content/SoulX-FlashHead/models/SoulX-FlashHead-1_3B'
WAV2VEC_PATH = '/content/SoulX-FlashHead/models/wav2vec2-base-960h'

# FPS de saida
OUTPUT_FPS = 25
# ─────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Configuracoes:')
print(f'  GPU           : {GPU_NOME}')
print(f'  Modo atencao  : {"SDPA (sem flash_attn)" if IS_TURING_CONSUMER else "flash_attn 2.8"}')
print(f'  Modelo        : SoulX-FlashHead Lite (1.3B)')
print(f'  Model path    : {MODEL_PATH}')
print(f'  Wav2vec path  : {WAV2VEC_PATH}')
print(f'  FPS saida     : {OUTPUT_FPS}')
print(f'  Saida         : {OUTPUT_DIR}')
print(f'  Drive usado   : NAO — tudo em /content/')
print('\n✅ Pronto! Execute a celula 12 para gerar.')

Configuracoes:
  GPU           : Tesla T4
  Modo atencao  : flash_attn 2.8
  Modelo        : SoulX-FlashHead Lite (1.3B)
  Model path    : /content/SoulX-FlashHead/models/SoulX-FlashHead-1_3B
  Wav2vec path  : /content/SoulX-FlashHead/models/wav2vec2-base-960h
  FPS saida     : 25
  Saida         : /content/outputs
  Drive usado   : NAO — tudo em /content/

✅ Pronto! Execute a celula 12 para gerar.


In [25]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 12 — GERAR O VÍDEO
# ═══════════════════════════════════════════════════════════════
import os, time, subprocess

%cd /content/SoulX-FlashHead

# Nome do arquivo de saida com timestamp
import time as t
ts = int(t.time())
OUTPUT_VIDEO = f'{OUTPUT_DIR}/flashhead_lite_{ts}.mp4'

cmd = [
    'python', 'generate_video.py',
    '--model_type',    'lite',
    '--ckpt_dir',      MODEL_PATH,
    '--wav2vec_dir',   WAV2VEC_PATH,
    '--cond_image',    SOURCE_IMAGE,   # <-- mudou
    '--audio_path',    AUDIO_WAV,
    '--save_file',     OUTPUT_VIDEO,   # <-- mudou
]

print(f'Gerando video com SoulX FlashHead Lite...')
print(f'Comando: {" ".join(cmd)}')
print('-' * 55)

inicio = time.time()

result = subprocess.run(cmd, capture_output=True, text=True)

elapsed = time.time() - inicio

print("STDOUT:\n", result.stdout)
print("STDERR:\n", result.stderr)
print("Return code:", result.returncode)

print('-' * 55)
print(f'Tempo total: {elapsed:.1f}s ({elapsed/60:.1f} min)')

# Verificar resultado — pode ter salvo com nome diferente
videos = sorted(
    [os.path.join(OUTPUT_DIR, f) for f in os.listdir(OUTPUT_DIR)
     if f.endswith(('.mp4', '.avi'))],
    key=os.path.getmtime, reverse=True
)

if videos:
    ULTIMO_VIDEO = videos[0]
    mb = os.path.getsize(ULTIMO_VIDEO) / (1024*1024)
    print(f'\n✅ Video gerado: {os.path.basename(ULTIMO_VIDEO)} ({mb:.1f} MB)')
    print('   Execute a celula 13 para visualizar e baixar.')
elif result != 0:
    ULTIMO_VIDEO = None
    print('\n⚠️  Erro na geracao. Verifique os logs acima.')
    print('   Solucoes comuns:')
    print('   1. Se o erro mencionar flash_attn: a celula 8 pode nao ter patchado tudo.')
    print('      Tente rodar novamente a celula 8 e depois esta celula.')
    print('   2. Se OOM (out of memory): reduza a resolucao da imagem para 256x256.')
    print('   3. Se KeyError no modelo: verifique se o download (celula 7) completou.')

/content/SoulX-FlashHead
Gerando video com SoulX FlashHead Lite...
Comando: python generate_video.py --model_type lite --ckpt_dir /content/SoulX-FlashHead/models/SoulX-FlashHead-1_3B --wav2vec_dir /content/SoulX-FlashHead/models/wav2vec2-base-960h --cond_image /content/inputs/avatar.png --audio_path /content/inputs/audio.wav --save_file /content/outputs/flashhead_lite_1776973059.mp4
-------------------------------------------------------
STDOUT:
 [generate] model denoise per step: 73.89895796775818s
[generate] model denoise per step: 1.3986783027648926s
[generate] model denoise per step: 1.409834384918213s
[generate] model denoise per step: 1.4061219692230225s
[generate] decode video frames: 25.04877996444702s
[generate] color correction: 0.0546717643737793s
[generate] encode motion frames: 1.099156379699707s
[generate] model denoise per step: 1.4134438037872314s
[generate] model denoise per step: 1.4234638214111328s
[generate] model denoise per step: 1.4182450771331787s
[generate] mod

In [11]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 13 — Visualizar e baixar o vídeo
# NAO salva no Google Drive — baixa direto para o seu PC
# ═══════════════════════════════════════════════════════════════
from google.colab import files
from IPython.display import HTML, display
from base64 import b64encode
import os

videos = sorted(
    [os.path.join(OUTPUT_DIR, f) for f in os.listdir(OUTPUT_DIR)
     if f.endswith('.mp4')],
    key=os.path.getmtime, reverse=True
)

if not videos:
    print('⚠️  Nenhum video encontrado. Execute a celula 12 primeiro.')
else:
    video_path = videos[0]
    nome = os.path.basename(video_path)
    mb = os.path.getsize(video_path) / (1024*1024)

    print(f'Reproduzindo: {nome} ({mb:.1f} MB)')
    print('-' * 45)

    with open(video_path, 'rb') as f:
        b64 = b64encode(f.read()).decode()

    display(HTML(f'''
    <video width="640" controls style="border-radius:8px;margin:10px 0">
        <source src="data:video/mp4;base64,{b64}" type="video/mp4">
    </video>
    <p style="font-family:monospace;color:#555">
        📹 {nome} &nbsp;|&nbsp; {mb:.1f} MB &nbsp;|&nbsp;
        Modelo: SoulX FlashHead Lite &nbsp;|&nbsp; GPU: {GPU_NOME}
    </p>
    '''))

    print('\nBaixando o video para o seu PC...')
    files.download(video_path)
    print('✅ Download iniciado! Verifique sua pasta de Downloads.')
    print('\n💡 Nada foi salvo no Google Drive — so no seu PC.')

⚠️  Nenhum video encontrado. Execute a celula 12 primeiro.


---
## 🔄 Gerar novo vídeo na mesma sessão

Execute **somente as células 9, 10 e 12** com novos arquivos.
Modelos e dependências já estão carregados.

---
## ⚡ Diferença deste modelo vs LivePortrait

| | LivePortrait | SoulX FlashHead Lite |
|---|---|---|
| **Tipo** | Warping 2D (estica pixels) | Difusão latente (regenera pixels) |
| **Pele esticada** | ❌ Sim, ocorre | ✅ Não — pixels novos gerados |
| **Lip sync** | ❌ Não nativo | ✅ Sim, sincroniza com áudio |
| **Velocidade T4** | ~12 FPS | ~15-20 FPS (Lite) |
| **Velocidade 4090** | 30+ FPS | 96 FPS |
| **VRAM** | ~3GB | ~6.4GB |

---
## 📋 Solução de problemas

| Problema | Solução |
|---|---|
| `ImportError: flash_attn` | Rode a célula 8 novamente (aplica o patch) |
| `CUDA out of memory` | Redimensione a imagem para 256x256 |
| Rosto não detectado | Foto frontal, bem iluminada, rosto grande |
| Audio fora de sincronia | Verifique se o audio esta em WAV 16kHz (célula 10 converte automaticamente) |
| Sessão expirou | Execute do início — células 1-7 levam ~15 min |

---
## 🔗 Links
- GitHub: https://github.com/Soul-AILab/SoulX-FlashHead
- HuggingFace: https://huggingface.co/Soul-AILab/SoulX-FlashHead-1_3B
- Demo online: https://huggingface.co/spaces/Soul-AILab/SoulX-FlashHead